# Modelling a non-stationary poisson process

>A non-stationary Poisson process (NSPP) is an arrival process where inter-arrival times are Exponentially distributed with a mean that varies by time.

One of the limitations of queuing theory is the difficulty of modelling time-dependent arrivals.  Computer simulation offers a number of ways of modelling non-stationary arrivals.  

🎓 **In this notebook you will learn:**

* ✅ How to implement the thinning algorithm to model a non-stationary poisson process (NSPP)
* ✅ How to check that that thinning is working correctly.
* 🎁 **BONUS**: How to improving the efficiency of your thinning algorithm code in Python. 


> **Special thanks** to two 2020/21 students Tamir and Simon who spotted bugs in the original code!

---

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import itertools
import simpy

# please use simpy version 4
simpy.__version__

----
## An example NSPP

The table below breaks an arrival process down into 60 minutes intervals.


| t(min) | Mean time between arrivals (min) | Arrival Rate $\lambda(t)$ (arrivals/min) |
|:------:|:--------------------------------:|:--------------------------------------:|
|    0   |                15                |                  1/15                  |
|   60   |                12                |                  1/12                  |
|   120  |                 7                |                   1/7                  |
|   180  |                 5                |                   1/5                  |
|   240  |                 8                |                   1/8                  |
|   300  |                10                |                  1/10                  |
|   360  |                15                |                  1/15                  |
|   420  |                20                |                  1/20                  |
|   480  |                20                |                  1/20                  |

> **Interpretation**: In the table above the fastest arrival rate is 1/5 customers per minute or 5 minutes between customer arrivals.

## Thinning

Thinning is a acceptance-rejection sampling method and is used to generate inter-arrival times from a NSPP.  

> A NSPP has arrival rate $\lambda(t)$ where $0 \leq t \leq T$

**The thinning algorithm**

A NSPP has arrival rate $\lambda(t)$ where $0 \leq t \leq T$

Here $i$ is the arrival number and $\mathcal{T_i}$ is its arrival time.

1. Let $\lambda^* = \max_{0 \leq t \leq T}\lambda(t)$ be the maximum of the arrival rate function and set $t = 0$ and $i=1$

2. Generate $e$ from the exponential distribution with rate $\lambda^*$ and let $t = t + e$ (this is the time of the next entity will arrive)

3. Generate $u$ from the $U(0,1)$ distribution.  If $u \leq \dfrac{\lambda(t)}{\lambda^*}$ then $\mathcal{T_i} =t$ and $i = i + 1$

4. Go to Step 2.

## Exercise 1: simulation **without thinning**

**Task:**
* Build a simple `simpy` model that simulates time-dependent arrivals
* For this exercise please **IGNORE** the need for a thinning process.

**Optional task:**
* It is useful to set the sampling of arrivals using a random seed.  This will allow you to compare the number of arrivals before and after adding thinning.  **Remember that an issue with DES without thinning occurs when moving from a period $t$ with a low arrival rate to $t+1$ that has a high one.**

**Hints:**
* Build your model up gradually. 
* Start by building a model that simulates exponential arrivals using a single mean inter-arrival time then add in logic to change which mean you use depending on the simulation time.
* The logic to decide the time period is equivalent to asking yourself "given `env.now()` and that arrival rates are split into 60 minute chunks which row of my dataframe should I select".
* To simplify the task you set the run length of the simulation to no more than 540 minutes.  For an extra challenge think about how you would run the model for longer than 480 minutes and loop back round to the first period (the code to do this is surprising simple).

The data are stored in a file `data/nspp_example1.csv`. 

In [ ]:
# your code here ...

## Exercise 2: Thinning the arrivals

**Task:**
* Update your exercise 1 code to include an implementation of thinning
* What do you notice about the total number of arrivals compared to the previous example? Why has the changed occurred?
   * If you are not controlling your sampling with random seeds you will need to run each implementation a few times.

**Hints:**
* You will need a second distribution - Uniform(0, 1) to do the thinning.  If you are controlling random sampling through seeds that means you will need a second seed.


In [ ]:
# your code here ...

## Exercise 3: Validate the total number of arrivals in 540 minutes.

Here we will repeat the simulation 10,000 times and then explore the distribution of the number of arrivals.  If all has gone to plan this should be a Poisson distribution with mean ~53.

**Task:**
* Run the code below substituting the name of your thinning function in the place holder.

**Hint**

* For each simulated run you need to count the number of arrivals. One option is to append the number of arrivals to a list and then after all 10,000 runs are complete plot a histogram (`plt.hist(list_name)`) and also take the mean of the list.

In [ ]:
# %%timeit # <- uncomment this line to run the code multiple times and evaluate efficiency

RUN_LENGTH = 540
REPLICATIONS = 10_000
audit = []

for i in range(REPLICATIONS):
    # set up audit for replication.
    env = simpy.Environment()

    seed_sequence = np.random.SeedSequence(i)
    seeds = seed_sequence.spawn(2)
    
    # env.process(thinning_function_name(thinning_params...) # <- update here
    
    env.run(RUN_LENGTH)

## **Optional** Exercise 4: Pre-calculate the acceptance probabilities

At the moment the code calculates the probability of accepting an arrival on each iteration. We can make the code more efficient by pre-calculating the acceptance probabilities before we run the function.

**Task**:

* Copy-paste your code and modify it so that it calculates the acceptance probabilities before entering the while Loop.
* Modify the loop so that the code looks up the acceptance probability rather than the mean IAT for that time period.

**Hints**:
* You can create a numpy array of acceptance probabilities like so `accept_probs =  (means['arrival_rate'] / lambda_max).to_numpy()`

In [ ]:
# your code here